# Transformers Explained — Part 2: Practical Python Implementation with Hugging Face
**DataOrbitAI** · Model: `distilgpt2` (a small, CPU-friendly, pretrained, decoder-only Transformer)

In Part 1 we studied the Transformer architecture. Here we open a real pretrained model and follow the text step by step:

**Text → Tokenization → Token IDs → Embeddings + Positional Information → Transformer Layers → Attention → Logits → Next-Token Generation**

> DistilGPT2 is a small educational model. It is **not** comparable to a modern production conversational LLM, and its text will often be repetitive or odd.
> Hugging Face Transformers is an open-source library; each pretrained model has its own license and usage conditions.

Run the cells from top to bottom. The first run downloads the model (about 350 MB).

## 1. Load the tokenizer and the model
- `AutoTokenizer` loads the tokenizer that matches the model. Tokenization is done by the tokenizer, **not** by the neural network.
- `AutoModelForCausalLM` loads the model with a language-modeling head. *Causal* means each position can only use earlier tokens.
- `eval()` switches to inference mode (turns dropout off).
- `attn_implementation="eager"` makes sure attention weights can be returned later.

**Expected output:** the `GPT2Config` with `n_layer=6`, `n_head=12`, `n_embd=768`, `vocab_size=50257`, `n_positions=1024`. Check the printed values on your machine.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager",
)

model.eval()

print(model.config)

## 2. Tokenization: text becomes numbers
A token is not necessarily a whole word. Each token maps to an integer **token ID**. Token IDs are only lookup indices, not embeddings. The **attention mask** marks which positions are real tokens (1) and which are padding (0).

**Expected output:** a list of tokens, a `(1, N)` tensor of IDs, and a `(1, N)` mask of ones (one sentence, no padding).

In [ ]:
text = "Transformers are powerful models for language understanding."

tokens = tokenizer.tokenize(text)

input_ids = tokenizer(
    text,
    return_tensors="pt"
)

print("Tokens:", tokens)
print("Token IDs:", input_ids["input_ids"])
print("Attention mask:", input_ids["attention_mask"])

## 3. Embeddings and positional information
The embedding table `wte` turns each token ID into a dense vector. The model also needs token **order**, so `wpe` adds a vector for each position.

> GPT-2-family models use **learned** positional embeddings. Other Transformer architectures use other schemes (sinusoidal, rotary, relative, …).

**Shape rule:** `(batch size, sequence length, hidden dimension)` → `(1, N, 768)` for both tensors.

`Input representation = Token embedding + Positional embedding`

In [ ]:
with torch.no_grad():
    token_embeddings = model.transformer.wte(
        input_ids["input_ids"]
    )

print("Token embedding shape:", token_embeddings.shape)

In [ ]:
position_ids = torch.arange(
    input_ids["input_ids"].size(1)
).unsqueeze(0)

with torch.no_grad():
    positional_embeddings = model.transformer.wpe(
        position_ids
    )

print("Positional embedding shape:", positional_embeddings.shape)

# Input representation = Token embedding + Positional embedding
input_representation = token_embeddings + positional_embeddings
print("Input representation shape:", input_representation.shape)

## 4. The Transformer architecture in code
`model.transformer.h` is a stack of identical Transformer blocks. Each block contains layer normalization, multi-head self-attention, and a feed-forward network (MLP), with residual connections.

**Expected output:** the three config values, then a `ModuleList` of 6 blocks (`GPT2Block`).

In [ ]:
print("Number of Transformer layers:", model.config.n_layer)
print("Number of attention heads:", model.config.n_head)
print("Hidden dimension:", model.config.n_embd)

print(model.transformer.h)

## 5. Hidden states and attention weights
- **Hidden states**: the token vectors after the embeddings and after every layer (so `n_layer + 1` tensors).
- **Attention weights**: for each layer, a tensor of shape `(batch size, number of heads, sequence length, sequence length)`. Row *i* shows how much position *i* looks at each earlier position. Because attention is **causal**, everything above the diagonal is 0.

> Attention weights show one part of the computation. They are **not** a complete explanation of the model's reasoning.

In [ ]:
with torch.no_grad():
    outputs = model(
        **input_ids,
        output_attentions=True,
        output_hidden_states=True
    )

print(
    "Number of hidden-state outputs:",
    len(outputs.hidden_states)
)

print(
    "Number of attention outputs:",
    len(outputs.attentions)
)

print(
    "First attention tensor shape:",
    outputs.attentions[0].shape
)

In [ ]:
# One attention head as numbers (layer 0, head 0). Rows sum to 1.
attn = outputs.attentions[0][0, 0]
print(attn.numpy().round(2))

In [ ]:
# OPTIONAL: heatmap (needs: pip install matplotlib)
try:
    import matplotlib.pyplot as plt
    labels = tokens
    plt.figure(figsize=(6, 5))
    plt.imshow(attn.numpy(), cmap="viridis")
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)
    plt.xlabel("Token being looked at")
    plt.ylabel("Token doing the looking")
    plt.title("Layer 0, head 0 attention")
    plt.colorbar()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed - skipping the optional heatmap.")

## 6. Logits and next-token prediction
**Logits** are raw scores, one per vocabulary token, for every sequence position: shape `(batch size, sequence length, vocabulary size)`. For next-token prediction we only need the **last** position. **Softmax** turns logits into probabilities; **argmax** picks the highest-scoring token (greedy decoding). The result is a token **ID**, which we decode back to text.

Other decoding strategies (sampling, temperature, top-k, top-p) can produce different outputs.

In [ ]:
with torch.no_grad():
    outputs = model(**input_ids)

logits = outputs.logits

print("Logits shape:", logits.shape)

next_token_logits = logits[:, -1, :]

next_token_id = torch.argmax(
    next_token_logits,
    dim=-1
)

print(
    "Predicted next token:",
    tokenizer.decode(next_token_id)
)

In [ ]:
# Logits are raw scores. Softmax turns them into probabilities.
probabilities = torch.softmax(next_token_logits, dim=-1)
top_probs, top_ids = torch.topk(probabilities, k=5, dim=-1)
print("Top 5 candidates for the next token:")
for prob, token_id in zip(top_probs[0], top_ids[0]):
    print(
        f"  id={token_id.item():>6}  "
        f"token={tokenizer.decode(token_id)!r:<14} "
        f"probability={prob.item():.4f}"
    )

## 7. Generate text
`generate()` repeats one loop: predict the next token → append it to the sequence → feed the longer sequence back in. This is **autoregressive** generation. With `do_sample=False` it is greedy and deterministic.

Quality depends on the model, its training data, the prompt, and the decoding strategy.

In [ ]:
prompt = "Data engineering is evolving because"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(
    tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )
)

### The same loop, one token at a time
This is what `generate()` does internally (greedy case, without caching).

In [ ]:
current_ids = inputs["input_ids"]
for step in range(1, 11):
    with torch.no_grad():
        step_logits = model(input_ids=current_ids).logits
    new_id = torch.argmax(step_logits[:, -1, :], dim=-1, keepdim=True)
    current_ids = torch.cat([current_ids, new_id], dim=-1)
    print(f"Step {step:>2}: new token = {tokenizer.decode(new_id[0])!r}")

print()
print(tokenizer.decode(current_ids[0], skip_special_tokens=True))

n_prompt = inputs["input_ids"].size(1)
print(
    "Matches generate() for these 10 tokens:",
    torch.equal(current_ids[0], generated[0][: n_prompt + 10]),
)

## Recap
Text → Tokenization → Token IDs → Embeddings + Positional Information → Transformer Layers → Attention → Logits → Next-Token Generation.

Next: embeddings → semantic search → vector databases → Retrieval-Augmented Generation (RAG) → LLM serving. Those get their own videos.

To save all real results from your machine as JSON, run `python capture_outputs.py`.